# Initialization

In [ ]:
import os
os.environ['CURL_CA_BUNDLE'] = ''
os.environ["WANDB_DISABLED"] = "true"
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datetime import datetime
import requests
import sys

# Add root folder into paths
sys.path.append(os.path.abspath('..'))

# Now we can import our own classes/functions
from utils.datahandling import format_example_for_training
from utils.modeling import prepare_model_for_training, generate_response

# Do this to ignore SSL errors (only apply if source is trusted)
response = requests.get("https://huggingface.co/api/whoami-v2", verify=False)

In [ ]:
instructions_path = '../instructions/v1 LLM instructions.txt'
dataset_path = '../datasets/dataset_v3.json'

# Read the instructions that will be given to the model
with open(instructions_path, 'r', encoding='utf-8') as file:
    system_instructions = file.read()

In [ ]:
# Set paths based on variables
model_type = 'Meltemi'
extra_info = ''

# Current date and time as name, if no name was given
if extra_info == '':
    now = datetime.now()
    extra_info = now.strftime(" %Y-%m-%d %H:%M:%S")

if model_type == 'Meltemi':
    model_id = "ilsp/Meltemi-7B-Instruct-v1.5"
    save_path = "../saved_models/meltemi" + extra_info
    output_dir = "../saved_models/outputs meltemi" + extra_info
elif model_type == 'Llama':
    model_id = "meta-llama/Llama-3.2-3B-Instruct"
    save_path = "../saved_models/llama" + extra_info
    output_dir = "../saved_models/outputs llama" + extra_info
elif model_type == 'Krikri':
    model_id = 'ilsp/Llama-Krikri-8B-Instruct'
    save_path = "../saved_models/krikri" + extra_info
    output_dir = "../saved_models/outputs krikri" + extra_info
else:
    raise ValueError("Invalid model type")

# Set up GPU memory optimization
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

# Data Preparation

In [ ]:
# Load data from JSON file
with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Alter messages into the format the model expects to see
messages = [format_example_for_training(datum['input'], datum['output'], system_instructions) for datum in data]

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Ensure the tokenizer has pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Prepare the dataset in a format suitable for instruction fine-tuning
prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

IGNORE_TOKEN_ID = -100  # Used by most HuggingFace models to ignore loss

train_data = []

for prompt in prompts:
    # Encode text to tokens
    encoding = tokenizer(prompt, padding='max_length', truncation=True, max_length=512, return_tensors='pt')
    input_ids = encoding["input_ids"].squeeze()
    attention_mask = encoding["attention_mask"].squeeze()

    # Find where the generation (response) should start
    response_start = prompt.find("<|assistant|>") + len("<|assistant|>")
    response_tokens = tokenizer(prompt[response_start:], padding='max_length', truncation=True, max_length=512, return_tensors='pt')["input_ids"].squeeze()

    # Build labels: mask everything before the assistant's response
    labels = input_ids.clone()
    num_response_tokens = (response_tokens != tokenizer.pad_token_id).sum()
    num_total_tokens = len(input_ids)
    response_start_token_idx = num_total_tokens - num_response_tokens
    labels[:response_start_token_idx] = IGNORE_TOKEN_ID

    # Update train data list
    train_data.append({
        "input_ids": input_ids.tolist(),
        "attention_mask": attention_mask.tolist(),
        "labels": labels.tolist(),
    })

# Create a Dataset object
train_dataset = Dataset.from_list(train_data)

# Model Training

In [ ]:
model = prepare_model_for_training(model_id)

In [ ]:
# Define DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=10,
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    fp16=True,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    remove_unused_columns=False,
)

In [ ]:
# Ensure model is on the correct device (Trainer will usually handle this, but PEFT sometimes needs explicit move)
device = "cuda" if torch.cuda.is_available() else "cpu"
if hasattr(model, "to") and callable(getattr(model, "to")):
    model = model.to(device)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

# Train the model
trainer.train()

# Save the fine-tuned model
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Inference Test

In [ ]:
# Try the model
print("Καλώς ήρθατε στον βοηθό ιατρικών ραντεβού. Πληκτρολογήστε 'έξοδος' για να τερματίσετε.")
while True:
    user_input = input("\nΗ ερώτησή σας: ")
    if user_input.lower() in ['έξοδος', 'exit', 'quit']:
        break

    response = generate_response(user_input, system_instructions, tokenizer, model)
    print("\n" + response, flush=True)